# Notebook 7: Sentence Transformers for Semantic Similarity

**Module 2 - Text Similarity**
*Author: Axel Sirota, Data Trainers LLC*

## The Scenario

You have word embeddings from Notebooks 5 and 6: one vector per word. But almost every real product works at the sentence or document level - FAQ matching, duplicate-question detection, semantic search, chatbot intent classification, RAG pipelines. Averaging word vectors throws away syntax and order: *"dog bites man"* and *"man bites dog"* end up with identical mean vectors.

In this notebook we produce one vector per *sentence* that preserves meaning. We compare three approaches - (1) a mean-of-GloVe baseline, (2) Sentence-BERT (SBERT), and (3) the legacy Doc2Vec model - and build a working semantic search engine over Yelp reviews. By the end you should be comfortable reaching for `all-MiniLM-L6-v2` as the default sentence encoder for most tasks.

## Learning objectives

By the end of this notebook you will be able to:

1. Explain why word-level embeddings are not enough for sentence-level tasks.
2. Compare three approaches to sentence representation: mean of word vectors, SBERT, Doc2Vec.
3. Use the `sentence-transformers` library: load a model, call `.encode()`, inspect output shapes.
4. Pick the right SBERT model from the HuggingFace hub (speed vs. quality trade-off).
5. Build a semantic search: encode the corpus once, encode the query, rank by cosine similarity.
6. Understand the difference between **symmetric** (similarity) and **asymmetric** (query/doc) retrieval.
7. Cluster sentences by embedding and inspect cluster quality.

## Prerequisites

- Notebook 5 (CBOW) and Notebook 6 (GloVe fine-tuning) for embedding intuition.
- Cosine similarity (Notebook 3).
- Comfortable with `pandas`, `numpy`, basic PyTorch tensors.

## Runtime

Recommended: **Colab with GPU**. Without GPU, encoding 2,000 reviews takes ~2 minutes; with GPU, ~10 seconds. We have a smaller fallback corpus too.


## Section 0: Environment Setup

Install dependencies. On a fresh Colab session this takes ~3-5 minutes the first time (it pulls `torch`, `transformers`, and `sentence-transformers` together). Subsequent runs in the same session are instant.

> **Note.** The very first call to `SentenceTransformer('all-MiniLM-L6-v2')` later in the notebook downloads ~80 MB of model weights from HuggingFace. Budget a few extra seconds for that.


In [ ]:
# Install required packages.
# - sentence-transformers >= 2.7 brings util.cos_sim and util.semantic_search.
# - gensim is used ONLY for the legacy Doc2Vec comparison at the end.
# - The rest are standard data-science staples.
!pip install -q "sentence-transformers>=2.7" "torch>=2.1" "scikit-learn>=1.3" \
    "gensim>=4.3" "pandas>=2.0" "matplotlib>=3.7" "seaborn>=0.13" "textblob>=0.17"


In [ ]:
# Imports: visualization, data, model, training (in that order).
import os
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

# Reproducibility: seed every RNG we touch.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device detection: SBERT will use CUDA automatically if available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

print("\nEnvironment ready.")


In [ ]:
# Hyperparameters for the whole notebook: defined up front, reused everywhere.
SEED          = 42
CORPUS_SIZE   = 2000          # Yelp reviews to use; lower to 500 if CPU-only.
SBERT_MODEL   = "all-MiniLM-L6-v2"   # 384-dim, ~80MB, fast. Default symmetric model.
# Alternative (better but heavier): "all-mpnet-base-v2" (768-dim, ~420MB)
# Alternative (Q/A asymmetric):     "multi-qa-MiniLM-L6-cos-v1"
BATCH_SIZE    = 64            # Encode in batches of 64: much faster than one-by-one.
TOP_K         = 5             # How many results to return for each search query.
N_CLUSTERS    = 5             # KMeans clusters for the clustering section.

print(f"Will encode up to {CORPUS_SIZE} reviews with model '{SBERT_MODEL}'.")


## Section 1: Why Sentence Embeddings?

Pretend you are building search for a help center. A user types:

> *"my package never arrived"*

Your knowledge base contains an article titled:

> *"Tracking a missing shipment"*

There is **zero word overlap** between query and article. TF-IDF and bag-of-words search will fail. Word embeddings alone won't save you either: you need to compose word vectors into a *sentence vector* that represents the meaning of the whole input.

### The naive baseline: average the word vectors

The simplest sentence embedding is just the mean of the GloVe vectors of its words:

```python
def sentence_embedding_mean(text, glove):
    tokens = text.lower().split()
    vecs = [glove[t] for t in tokens if t in glove]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)
```

This sometimes works. Two failures show its limits and motivate everything that follows in this notebook:

1. **Word order is destroyed.** *"dog bites man"* and *"man bites dog"* contain the exact same three words, so they get identical mean vectors. The model cannot tell newsworthy from boring.
2. **Antonyms look identical.** *"I love it"* and *"I hate it"* differ in *one* word. With mean-of-GloVe their cosine similarity is ~0.95. The model does not understand polarity.

We will reproduce both failures below, then watch SBERT fix them.


In [ ]:
# Download GloVe 100d (same file Notebook 6 uses) and build a {word: vector} dict.
# This file is ~330MB. The download happens once per Colab session.
import urllib.request

GLOVE_PATH = "./glove.6B.100d.txt"
GLOVE_URL  = "https://www.dropbox.com/s/dl1vswq2sz5f1ws/glove.6B.100d.txt?dl=1"

if not os.path.exists(GLOVE_PATH):
    print("Downloading GloVe 6B 100d (~330MB) ...")
    urllib.request.urlretrieve(GLOVE_URL, GLOVE_PATH)
    print("Done.")

glove = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        glove[parts[0]] = np.asarray(parts[1:], dtype=np.float32)

print(f"Loaded {len(glove):,} GloVe vectors of dim {len(next(iter(glove.values())))}")


In [ ]:
# A tiny helper that turns a sentence into a mean-of-GloVe vector.
# Returns shape (100,). If no token is in the vocab, returns zeros.

def sentence_embedding_mean(text, glove_dict, dim=100):
    tokens = text.lower().split()
    vecs = [glove_dict[t] for t in tokens if t in glove_dict]
    if not vecs:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

# Quick sanity check.
v = sentence_embedding_mean("the food was great", glove)
print(f"Vector shape: {v.shape}")
print(f"First 5 dims: {v[:5]}")


In [ ]:
# The two failure cases that motivate SBERT.
def cos(u, v):
    """Tiny cosine similarity over numpy 1-D arrays."""
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12))

pairs = [
    ("dog bites man",            "man bites dog"),            # Word order
    ("I love it",                "I hate it"),                 # Antonym
    ("the food was excellent",   "the meal was outstanding"),  # True paraphrase
    ("the food was excellent",   "my car broke down"),         # Unrelated
    ("the chef was attentive",   "the server was friendly"),   # Loose paraphrase
]

print(f"{'Sentence A':<35}{'Sentence B':<35}{'Mean-GloVe cos':>18}")
print("-" * 88)
for a, b in pairs:
    va = sentence_embedding_mean(a, glove)
    vb = sentence_embedding_mean(b, glove)
    print(f"{a:<35}{b:<35}{cos(va, vb):>18.4f}")


### What just happened?

Look at the table above. You should see something like:

- *"dog bites man"* vs *"man bites dog"*: cosine = **1.00** (literally identical, order destroyed).
- *"I love it"* vs *"I hate it"*: cosine is about **0.95** (antonyms look near-identical because everything except one word matches).
- True paraphrases also score high, but so do the antonyms, so the model has no discrimination power.

This is why production systems do **not** ship mean-of-word-vectors as a sentence representation. We need an encoder that understands composition, which is what SBERT provides in the next section.


## Section 2: Sentence-BERT (SBERT)

[Reimers & Gurevych, EMNLP 2019](https://arxiv.org/abs/1908.10084) introduced **Sentence-BERT**: take a pretrained BERT, attach a pooling layer (mean-pool the token vectors), then **fine-tune the whole stack** with a *siamese* setup on labeled sentence pairs (NLI, STS, paraphrase data). The result: a single forward pass turns a sentence into a fixed-dim vector that lives in a space where cosine similarity corresponds to semantic similarity.

### Architecture in one diagram

```
       sentence A                     sentence B
           |                              |
        [BERT]   ←  shared weights →  [BERT]
           |                              |
       mean-pool                      mean-pool
           |                              |
         u (768d)                      v (768d)
           \____________   ____________/
                        \ /
              cosine(u, v)  →  loss vs. label
```

At inference, you only need *one* BERT pass per sentence (no pairs needed).

### The model zoo

| Model | Dim | Size | Best for |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | 80 MB | **Default.** Fast, quality is excellent. |
| `all-mpnet-base-v2` | 768 | 420 MB | Higher quality, ~3x slower. |
| `multi-qa-MiniLM-L6-cos-v1` | 384 | 80 MB | **Asymmetric** query-to-doc retrieval. |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | 470 MB | 50+ languages. |

For this notebook we will use `all-MiniLM-L6-v2`. It is the right default for >90% of use cases.


In [ ]:
# Load the SBERT model. First call downloads ~80MB from HuggingFace and caches
# it under ~/.cache/torch/sentence_transformers (or /root/.cache/... in Colab).
print(f"Loading {SBERT_MODEL} ...")
t0 = time.time()
sbert = SentenceTransformer(SBERT_MODEL, device=device)
print(f"Loaded in {time.time() - t0:.1f}s")
print(f"Embedding dimension: {sbert.get_sentence_embedding_dimension()}")
print(f"Max input tokens   : {sbert.max_seq_length}")


In [ ]:
# First encoding call. Notice the API: list in, ndarray out.
sentences_demo = ["I love pizza", "pizza is great"]

emb_demo = sbert.encode(
    sentences_demo,
    convert_to_numpy=True,    # numpy is friendlier for sklearn / pandas
    show_progress_bar=False,
)
print(f"Output shape: {emb_demo.shape}")        # (2, 384)
print(f"Dtype       : {emb_demo.dtype}")
print(f"L2 norm of first vector: {np.linalg.norm(emb_demo[0]):.4f}")


In [ ]:
# Redo the same five pairs from the GloVe baseline, but now with SBERT.
sbert_pairs = sbert.encode([a for a, _ in pairs] + [b for _, b in pairs],
                           convert_to_numpy=True, show_progress_bar=False)
embA = sbert_pairs[: len(pairs)]
embB = sbert_pairs[len(pairs):]

print(f"{'Sentence A':<35}{'Sentence B':<35}{'GloVe cos':>12}{'SBERT cos':>12}")
print("-" * 94)
for i, (a, b) in enumerate(pairs):
    g = cos(sentence_embedding_mean(a, glove), sentence_embedding_mean(b, glove))
    s = float(util.cos_sim(embA[i], embB[i]))
    print(f"{a:<35}{b:<35}{g:>12.4f}{s:>12.4f}")


### Read the table above carefully

You should see the antonym pair (*"I love it"* vs *"I hate it"*) drop from ~0.95 with mean-of-GloVe to something noticeably lower with SBERT, typically around 0.5-0.7. The word-order pair stays similar (both sentences are about a dog and a man biting), but the true paraphrase ("food was excellent" vs "meal was outstanding") scores higher.

> **Caveat.** Paraphrase-tuned is not the same as polarity-aware. SBERT is trained on paraphrase and NLI data, so it captures *topical* similarity very well. It is less sensitive to polarity than a sentiment-tuned model would be. If your job is sentiment classification, do not stop here: fine-tune a sentiment model. We come back to this in Modules 3 and 4.


In [ ]:
# Why batching matters: encode 200 sentences one-by-one vs. in a single batch.
fake_corpus = [f"this is sentence number {i} about food and service" for i in range(200)]

# One at a time.
t0 = time.time()
_ = [sbert.encode(s, show_progress_bar=False) for s in fake_corpus]
slow = time.time() - t0

# Batched.
t0 = time.time()
_ = sbert.encode(fake_corpus, batch_size=BATCH_SIZE, show_progress_bar=False)
fast = time.time() - t0

print(f"One-by-one : {slow:6.2f}s")
print(f"Batched    : {fast:6.2f}s   ({slow / max(fast, 1e-6):.1f}x faster)")


### Lab 1. Polarity Heatmap

Encode 20 sentences (10 positive and 10 negative Yelp reviews), then plot the 20x20 cosine-similarity heatmap. Do same-polarity pairs cluster?

**Steps:**

1. From the Yelp DataFrame `yelp` (loaded later; for this lab grab the first 10 of each polarity now from the small sample), build a list `mini_corpus` of 20 short reviews: 10 with `stars == 5` and 10 with `stars == 1`. Truncate each to ~200 chars for readability.
2. Encode them with `sbert.encode(...)` into `mini_emb` (use `convert_to_numpy=True`).
3. Compute the 20x20 cosine-similarity matrix `sim` with `cosine_similarity` from sklearn.
4. Plot it as a heatmap with `sns.heatmap`. Use `vmin=0`, `vmax=1`, and a diverging colormap.
5. Look at the resulting blocks: do you see a clear `pos | neg` block structure?

> **Note.** The Yelp dataset is loaded later in the notebook (Section 3). For this lab we need it now, so the next code cell loads a mini sample independently.


In [ ]:
# Quick mini-loader, independent of the bigger corpus we build in Section 3.
import urllib.request
if not os.path.exists("./yelp.csv"):
    urllib.request.urlretrieve("https://www.dropbox.com/s/xds4lua69b7okw8/yelp.csv?dl=1", "./yelp.csv")

_yelp_mini = pd.read_csv("./yelp.csv")
_yelp_mini = _yelp_mini[(_yelp_mini.stars == 5) | (_yelp_mini.stars == 1)].sample(frac=1, random_state=SEED)
positive = _yelp_mini[_yelp_mini.stars == 5].head(10)["text"].tolist()
negative = _yelp_mini[_yelp_mini.stars == 1].head(10)["text"].tolist()
print(f"Got {len(positive)} positive + {len(negative)} negative reviews.")


In [ ]:
# Lab 1: Polarity heatmap. Fill in the four steps below.

# 1. Build a 20-sentence corpus, truncated for readability.
mini_corpus = None  # YOUR CODE  (hint: list comprehension over positive + negative, [:200])

# 2. Encode with SBERT into a numpy array.
mini_emb = None     # YOUR CODE  (hint: sbert.encode(..., convert_to_numpy=True))

# 3. Compute the 20x20 cosine-similarity matrix.
sim = None          # YOUR CODE  (hint: sklearn cosine_similarity)

# 4. Plot the heatmap. Label rows/cols 'P0..P9' and 'N0..N9' so you can see polarity blocks.
labels = [f"P{i}" for i in range(10)] + [f"N{i}" for i in range(10)]
if sim is not None:
    plt.figure(figsize=(10, 8))
    sns.heatmap(sim, vmin=0, vmax=1, cmap="coolwarm",
                xticklabels=labels, yticklabels=labels, square=True, cbar_kws={"label": "cosine"})
    plt.title("Pairwise SBERT cosine similarity (P=positive, N=negative)")
    plt.tight_layout()
    plt.show()


## Section 3: A Working Semantic Search Engine

This is the headline use case. The pipeline is simple:

**Offline (once):**
1. Encode every document in your corpus into a matrix `E` of shape `(N, D)`.

**Online (per query):**
1. Encode the query into a vector `q` of shape `(D,)`.
2. Compute cosine similarity between `q` and every row of `E`.
3. Return the top-K rows.

That is the core of how production vector stores like pgvector or FAISS work, minus the index data structure. For corpora up to ~1M documents on a single machine, brute-force cosine over a numpy matrix is fine. Above that, you reach for a dedicated vector index; the vectors themselves are the same.

We will use the Yelp reviews dataset (1-star and 5-star reviews) and ask things like *"the chef was attentive"*, where the exact words won't appear in any review.


In [ ]:
# Download the Yelp reviews dataset (the same one used elsewhere in the course).
YELP_PATH = "./yelp.csv"
YELP_URL  = "https://www.dropbox.com/s/xds4lua69b7okw8/yelp.csv?dl=1"

if not os.path.exists(YELP_PATH):
    print("Downloading Yelp reviews ...")
    urllib.request.urlretrieve(YELP_URL, YELP_PATH)
    print("Done.")

yelp = pd.read_csv(YELP_PATH)
yelp = yelp[(yelp.stars == 5) | (yelp.stars == 1)].reset_index(drop=True)
yelp = yelp.sample(n=min(CORPUS_SIZE, len(yelp)), random_state=SEED).reset_index(drop=True)
yelp["text"] = yelp["text"].astype(str).str.strip()

corpus = yelp["text"].tolist()
print(f"Corpus size: {len(corpus):,} reviews")
print(f"Star distribution: {dict(yelp['stars'].value_counts())}")
print(f"\nExample review:\n{corpus[0][:300]}...")


In [ ]:
# Encode the entire corpus. Time it so students see GPU vs. CPU difference.
t0 = time.time()
corpus_emb = sbert.encode(
    corpus,
    batch_size=BATCH_SIZE,
    convert_to_tensor=True,        # keep on GPU for fast cosine later
    show_progress_bar=True,
)
elapsed = time.time() - t0

print(f"\nEncoded {len(corpus):,} reviews in {elapsed:.1f}s")
print(f"Embedding tensor: shape={tuple(corpus_emb.shape)}, device={corpus_emb.device}, dtype={corpus_emb.dtype}")
print(f"Memory footprint: ~{corpus_emb.element_size() * corpus_emb.numel() / 1e6:.1f} MB")


In [ ]:
# A reusable search function.
def search(query, top_k=TOP_K, model=sbert, corpus_embeddings=corpus_emb, corpus_texts=corpus):
    """Encode `query`, score against `corpus_embeddings`, return top-K (score, text) pairs."""
    q_emb = model.encode(query, convert_to_tensor=True, show_progress_bar=False)
    # util.cos_sim handles the L2 normalization for us; returns shape (1, N).
    scores = util.cos_sim(q_emb, corpus_embeddings)[0]
    # torch.topk for efficient top-K on GPU.
    top = torch.topk(scores, k=top_k)
    return [(float(s), corpus_texts[int(i)]) for s, i in zip(top.values, top.indices)]

# Demo queries: none of them require literal word overlap.
demo_queries = [
    "the chef was attentive",
    "best service ever",
    "terrible food, would not return",
    "great value for the money",
]

for q in demo_queries:
    print(f"\n=== Query: {q!r}")
    for score, text in search(q, top_k=3):
        snippet = text[:160].replace("\n", " ")
        print(f"  [{score:.3f}] {snippet}...")


### Lab 2. Polarity-filtered semantic search

Augment `search()` to accept an optional `stars_filter` argument so callers can restrict the search to, say, only 5-star reviews when looking for *"best X"* or only 1-star reviews when looking for *"worst X"*.

**Steps:**

1. Write `search_filtered(query, stars_filter=None, top_k=5)`.
2. If `stars_filter` is given (e.g. `5` or `1`), restrict scoring to only reviews with that star rating. (Hint: `yelp["stars"].values == stars_filter` gives a boolean mask the same length as `corpus_emb`.)
3. Test with at least three queries, including one (e.g. *"unforgettable evening"*) where the literal words don't appear in any review.
4. Compare results against a TF-IDF baseline (already provided) for one of your queries. Which retrieves more relevant items?


In [ ]:
# Lab 2: Filtered semantic search.

stars_arr = yelp["stars"].values   # length-N int array, aligned with corpus_emb

def search_filtered(query, stars_filter=None, top_k=TOP_K):
    """Like search(), but optionally restrict to reviews with a given star rating."""
    q_emb = sbert.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, corpus_emb)[0]   # shape (N,)

    # 1. If stars_filter is set, mask out non-matching reviews by setting their score to -inf.
    if stars_filter is not None:
        mask = None  # YOUR CODE  (hint: torch.tensor(stars_arr == stars_filter, device=scores.device))
        # YOUR CODE: scores[~mask] = -float("inf")

    # 2. Top-K argmax.
    top = torch.topk(scores, k=top_k)
    return [(float(s), int(i), corpus[int(i)]) for s, i in zip(top.values, top.indices)]

# Test it.
queries_to_test = [
    ("best service",          5),
    ("worst customer service", 1),
    ("unforgettable evening", None),
]
for q, sf in queries_to_test:
    print(f"\n=== {q!r}  (filter={sf}) ===")
    for score, idx, text in search_filtered(q, stars_filter=sf, top_k=3):
        print(f"  [{score:.3f}] (stars={int(stars_arr[idx])}) {text[:160]}...")


In [ ]:
# Compare semantic search to a TF-IDF baseline on the same query.
tfidf = TfidfVectorizer(stop_words="english", max_features=20000)
tfidf_matrix = tfidf.fit_transform(corpus)

def tfidf_search(query, top_k=3):
    qv = tfidf.transform([query])
    scores = cosine_similarity(qv, tfidf_matrix)[0]
    top = scores.argsort()[::-1][:top_k]
    return [(float(scores[i]), corpus[i]) for i in top]

q = "the chef was attentive"
print(f"=== Query: {q!r}\n")

print("--- TF-IDF top-3 ---")
for s, t in tfidf_search(q):
    print(f"  [{s:.3f}] {t[:160]}...")

print("\n--- SBERT top-3 ---")
for s, t in search(q, top_k=3):
    print(f"  [{s:.3f}] {t[:160]}...")


TF-IDF needs literal word overlap. *"chef"* and *"attentive"* may simply not appear in many reviews, so TF-IDF returns scores near zero or grabs unrelated reviews that happen to contain one of those words. SBERT pulls reviews about attentive servers, helpful staff, and friendly chefs, even when the exact words don't match, which is what makes dense retrieval worth the extra work.


## Section 4: Clustering Sentences

Sentence embeddings open the door to **unsupervised** structure discovery. Two questions worth asking:

1. If we run KMeans on these vectors, do we recover meaningful topics, even though SBERT was never trained on this corpus?
2. Where does each review sit in 2-D? (We use t-SNE for visualization.)

This is how product analytics teams often pre-cluster support tickets or customer reviews before manual labeling.


In [ ]:
# KMeans on SBERT embeddings.
# We need the embeddings on CPU as numpy for sklearn.
emb_np = corpus_emb.cpu().numpy()

km = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init="auto")
clusters = km.fit_predict(emb_np)

# For each cluster, pull the 3 reviews closest to the centroid (most "central").
print(f"=== {N_CLUSTERS} clusters discovered ===\n")
for c in range(N_CLUSTERS):
    members = np.where(clusters == c)[0]
    if len(members) == 0:
        continue
    # Distance to centroid for each member.
    dists = np.linalg.norm(emb_np[members] - km.cluster_centers_[c], axis=1)
    closest = members[np.argsort(dists)[:3]]
    print(f"--- Cluster {c}  (n={len(members)}) ---")
    for idx in closest:
        snippet = corpus[idx][:200].replace("\n", " ")
        print(f"  • {snippet}...")
    print()


In [ ]:
# t-SNE visualization (slow on 2000 points but very informative).
print("Running t-SNE (may take ~30s) ...")
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, init="pca")
emb_2d = tsne.fit_transform(emb_np)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=clusters, cmap="tab10", s=12, alpha=0.7)
plt.legend(*scatter.legend_elements(), title="cluster", loc="best")
plt.title(f"SBERT embeddings of Yelp reviews: t-SNE, colored by KMeans (k={N_CLUSTERS})")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.tight_layout()
plt.show()


### Lab 3. Vary the number of clusters

Run KMeans with `k = 3, 5, 10` on `emb_np` and inspect the top-3 centroid-closest reviews per cluster for each `k`. Then briefly answer (in a markdown cell or as a print statement):

- Which `k` produced the most interpretable clusters in your opinion?
- Why might "more clusters" not always be "better"? (Hint: think about cluster purity vs. cluster coverage.)


In [ ]:
# Lab 3: Vary k and inspect.

# 1. Loop over k_values, fit KMeans, print top-3 reviews per cluster.
k_values = [3, 5, 10]

# YOUR CODE: loop over k_values, for each one fit KMeans on emb_np,
#            and print top-3 closest-to-centroid reviews per cluster.
for k in k_values:
    print(f"\n========== k = {k} ==========")
    # km_k = ...
    # clusters_k = ...
    # for c in range(k):
    #     members = ...
    #     dists = ...
    #     closest = ...
    #     print(...)
    pass  # YOUR CODE


## Section 5: What About Doc2Vec? (Legacy comparison)

Doc2Vec ([Le & Mikolov, 2014](https://arxiv.org/abs/1405.4053)) was the standard pre-SBERT answer to "give me one vector per document." It extends Word2Vec by adding a paragraph-level vector to the context window. Two flavors:

- **PV-DM** (Distributed Memory): predict a target word from `[paragraph_id] + surrounding_words`.
- **PV-DBOW** (Distributed Bag of Words): predict surrounding words from `paragraph_id` alone.

It still appears in legacy systems, so it's worth knowing. We train one on the same Yelp corpus and qualitatively compare a "most similar" query to SBERT.

> **Verdict.** SBERT wins on quality (paraphrase-aware via NLI fine-tuning), is roughly the same speed at inference once loaded, and is much better out-of-the-box (no training step on your corpus required). Pick Doc2Vec only if you cannot install `transformers` for some platform reason.


In [ ]:
# Train a small Doc2Vec model on the same Yelp corpus for an apples-to-apples comparison.
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# Doc2Vec wants a list of TaggedDocument(words=[...], tags=[id]).
tagged = [TaggedDocument(words=t.lower().split(), tags=[i]) for i, t in enumerate(corpus)]

t0 = time.time()
d2v = Doc2Vec(
    documents=tagged,
    vector_size=100,    # smaller than SBERT's 384, but Doc2Vec is shallow
    window=5,
    min_count=2,
    workers=2,          # workers=4 sometimes hangs on Colab
    epochs=20,
    seed=SEED,
)
print(f"Doc2Vec trained in {time.time() - t0:.1f}s")
print(f"Vocabulary size: {len(d2v.wv):,}")


In [ ]:
# Pick one review at random, find its 3 nearest neighbors with each model.
probe_idx = 7
probe_text = corpus[probe_idx]
print(f"=== Probe review (#{probe_idx}) ===")
print(probe_text[:300], "...\n")

# --- SBERT neighbors ---
sbert_scores = util.cos_sim(corpus_emb[probe_idx], corpus_emb)[0]
sbert_top = torch.topk(sbert_scores, k=4)  # top-4 because #1 is the probe itself
print("--- SBERT top-3 neighbors ---")
for score, idx in zip(sbert_top.values[1:], sbert_top.indices[1:]):
    print(f"  [{float(score):.3f}] {corpus[int(idx)][:180]}...")

# --- Doc2Vec neighbors ---
print("\n--- Doc2Vec top-3 neighbors ---")
for idx, score in d2v.dv.most_similar(probe_idx, topn=3):
    print(f"  [{score:.3f}] {corpus[int(idx)][:180]}...")


## Section 6: Homework / Capstone Mini-Lab

Pick a **different** dataset and rebuild the semantic search. Two options:

- **SMS Spam** ([dataset](https://www.dropbox.com/scl/fi/yy0b8tblxx787vw0ncbm4/SMSSpamCollection.tsv?rlkey=iuk84q9leb2hcyisuvqe4c1hn&dl=1)): short messages, two classes (spam / ham). Try queries like *"call this number now"* or *"meet me at the cafe"*.
- **BBC news** ([dataset](https://www.dropbox.com/scl/fi/lfa2ryv86uqd3y988irfw/bbc.csv?rlkey=vtwdf6g8sejhkf75p7o36ev00&dl=1)): longer documents, 5 categories. Try queries like *"central bank cuts interest rates"*.

**Required deliverables:**

1. Load and preview the dataset.
2. Encode the text column with `sbert.encode(...)`.
3. Reuse the `search()` function (just point it at the new corpus and embeddings).
4. Submit *3* queries you find interesting, with their top-3 results.
5. **Reflection (1 paragraph):** which queries did SBERT handle well? Where did it surprise you (positively or negatively)?

> **Note.** You do not need to retrain anything. Same model, new corpus is the whole point of pretrained sentence encoders.


## Section 7: Wrap-up

### What you can now ship

- A semantic search engine over any text corpus, in <50 lines of code, with quality that beats TF-IDF on natural-language queries.
- Sentence-level clustering for unsupervised topic discovery on customer reviews, support tickets, and similar data.
- An honest assessment of when SBERT is not enough (sentiment polarity, factual entailment) and what to reach for next.

### Key takeaways

| Concept | Default choice |
|---|---|
| General sentence embeddings | `all-MiniLM-L6-v2` |
| When you need higher quality | `all-mpnet-base-v2` |
| Query-to-doc retrieval (asymmetric) | `multi-qa-MiniLM-L6-cos-v1` |
| Multilingual | `paraphrase-multilingual-MiniLM-L12-v2` |
| Cosine similarity | `sentence_transformers.util.cos_sim` (handles L2 norm) |
| Top-K over big corpora (>1M docs) | production vector stores like pgvector or FAISS (vectors are the same) |

### Common pitfalls

1. **GPU vs CPU.** Encoding 2,000 sentences takes ~2 min on CPU, ~10 s on GPU. Use Colab GPU.
2. **`convert_to_tensor` vs `convert_to_numpy`.** Pick one consistently. Tensor is faster for downstream torch ops; numpy is friendlier for sklearn.
3. **Symmetric vs asymmetric retrieval.** `all-MiniLM-L6-v2` assumes both inputs are similar in length and style. For query-to-doc (short query, long doc), use `multi-qa-MiniLM-L6-cos-v1`.
4. **L2 normalization.** `util.cos_sim` handles it. If you do dot product manually, normalize first or your scores are wrong.
5. **Paraphrase is not polarity.** SBERT can rank *"I love this place"* and *"I hate this place"* as fairly similar (same topic). Fine-tune a sentiment classifier on top if you need polarity.

### Self-check

1. When would you choose `all-mpnet-base-v2` over `all-MiniLM-L6-v2`? And the reverse?
2. You have 1M documents and want sub-50ms search. Is brute-force cosine over all of them realistic per query? What do you reach for?
3. `most_similar` to *"I love this place"* returns *"I hate this place"* in the top 5. Why? What does this tell you about SBERT's training objective?

### What's next

Module 3 starts with a feed-forward MLP that uses embeddings to classify text. You'll see how the embeddings you've built here are the input layer for almost every downstream NLP model. After that, a full BERT fine-tune for the same task.
